In [1]:
import pathlib
import os
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
from torch.utils.tensorboard.writer import SummaryWriter
import trimesh

import network, utils, dataset, train

%load_ext autoreload
%autoreload 2

/home/nikola/miniconda3/envs/adlr/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M")
# denoiser_name = "Resnet_2Blocks_256Hidden"
denoiser_name = "Resnet_2Blocks_256Hidden_SNR_Loss_50epochs"
experiment_name = f"{current_time}_{denoiser_name}"


config = {
    'experiment_name': experiment_name,
    'device': 'cuda:0',
    'is_overfit': True,
    'batch_size': 32,
    'resume_ckpt': None,
    'learning_rate': 0.0005,
    'lambda': 0.05,
    'max_epochs': 50,
    'timesteps': 1000,
    'print_every_n': 1, # every n batches
    'validate_every_n': 25,
    'print_EMD_every_n': 1
}

In [4]:
# declare device
device = torch.device('cpu')
if torch.cuda.is_available() and config['device'].startswith('cuda'):
    device = torch.device(config['device'])
    print('Using device:', config['device'])
else:
    print('Using CPU')

# create dataloaders
trainset = dataset.Dataset('train' if not config['is_overfit'] else 'overfit', config['timesteps'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=config['batch_size'], shuffle=True, num_workers=1)

# valset = dataset.Dataset('val' if not config['is_overfit'] else 'overfit', config['timesteps'])
# valloader = torch.utils.data.DataLoader(valset, batch_size=config['batch_size'], shuffle=False, num_workers=1)

denoiser = network.Denoiser()
diffuser = network.Diffuser(config['timesteps'])

# load model if resuming from checkpoint
if config['resume_ckpt'] is not None:
        utils.reload_model(denoiser, diffuser, config['experiment_name'], device)

# move model to specified device
denoiser.to(device)
diffuser.to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=config['learning_rate'])

# Create tensorboard writer    
log_path = pathlib.Path(f"logs/diffusion_training/{datetime.datetime.now().strftime('%b%d')}/{config['experiment_name']}")
writer = SummaryWriter(log_path)

#Run this code in terminal to start tensorboard: tensorboard --logdir=diffusion/nikola/logs/diffusion_training


total, trainable = utils.count_parameters(denoiser)
print(f"Total: {total:,} | Trainable: {trainable:,} | Model size: {utils.model_memory_size(denoiser):.3f} MB")

Using device: cuda:0
Total: 824,067 | Trainable: 824,067 | Model size: 3.144 MB


In [17]:
# start training
#tensorboard --logdir=diffusion/nikola/logs/diffusion_training

train.train(denoiser=denoiser, diffuser=diffuser, trainloader=trainloader, device=device, optimizer=optimizer, config=config, writer=writer,valloader=None)

[00/000] train_loss: 0.957
[00/001] train_loss: 0.834
[00/002] train_loss: 0.699
[00/003] train_loss: 0.547
[00/004] train_loss: 0.391
[00/005] train_loss: 0.255
[00/006] train_loss: 0.210
[00/007] train_loss: 0.148
[00/008] train_loss: 0.170
[00/009] train_loss: 0.260
[00/010] train_loss: 0.137
[00/011] train_loss: 0.130
[00/012] train_loss: 0.138
[00/013] train_loss: 0.111
[00/014] train_loss: 0.113
[00/015] train_loss: 0.123
[00/016] train_loss: 0.117
[00/017] train_loss: 0.135
[00/018] train_loss: 0.107
[00/019] train_loss: 0.118
[00/020] train_loss: 0.111
[00/021] train_loss: 0.073
[00/022] train_loss: 0.097
[00/023] train_loss: 0.088
[00/024] train_loss: 0.124
[00/025] train_loss: 0.109
[00/026] train_loss: 0.075
[00/027] train_loss: 0.091
[00/028] train_loss: 0.109
[00/029] train_loss: 0.124
[00/030] train_loss: 0.118
[00/031] train_loss: 0.139
[01/000] train_loss: 0.174
[01/001] train_loss: 0.094
[01/002] train_loss: 0.084
[01/003] train_loss: 0.122
[01/004] train_loss: 0.109
[

In [ ]:
#Load a model:
denoiser_id = "May13_14-53_Resnet_2Blocks_256Hidden_reweighted_loss_10000_epochs"
utils.reload_model(denoiser, diffuser, denoiser_id, device)

In [8]:
generateDDIM = True
generateDDPM = False
number_of_points = 2056
DDIM_steps = 200
number_of_DDIM_iterations = 1

if generateDDIM:
    for _ in range(number_of_DDIM_iterations):
        generated_pc_ddim = network.sample_ddim(denoiser, diffuser, n_points=number_of_points, steps=DDIM_steps)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(trainset[0], generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")

if generateDDPM:
    generated_pc_ddpm = network.sample_ddpm(denoiser, diffuser, n_points=number_of_points)
    generated_pcd_ddpm = trimesh.PointCloud(generated_pc_ddpm.squeeze().cpu().numpy())
    utils.visualize_comparison(trainset[0], generated_pc_ddpm, window_name="DDPM Target (Red) vs Generated (Blue)")

Visualizing: Target is RED, Generated is BLUE.


In [ ]:
generated_pc, samples_list = network.sample_and_capture(denoiser, diffuser, n_points=number_of_points, save_every=10)
utils.visualize_diffusion_progress(samples_list, window_name="Diffusion Process")

In [ ]:
# Optionally, save the generated point clouds to disk
generated_pcd_ddim.export(f"output/{denoiser_id}_ddim.obj")
generated_pcd_ddpm.export(f"output/{denoiser_id}_ddpm.obj")